# TAP access to SOHO metadata

To install [astroquery](https://astroquery.readthedocs.io/en/latest/index.html) ([Ginsburg et al, 2019](https://doi.org/10.3847/1538-3881/aafc33)) to be able to use [TAP+](https://astroquery.readthedocs.io/en/latest/utils/tap.html):

`conda install -c astropy astroquery`

The different missions have their own TAP servers; SOHO's is:
`https://ssa.esac.esa.int/ssa-sl-tap/data? `

## Tables: metadata of metadata

First of all, it's helpful to know what tables are there. 

From example 1.1 on [Astroquery TAP page](https://astroquery.readthedocs.io/en/latest/utils/tap.html), load the table objects using the method `load_tables()`, then print the names of those table objects using the `get_qualified_name()` method. This returns the schema (e.g., csa.) and the table names (e.g., csa.dataset). We only need to focus on the `csa.` schema tables.

In [6]:
from astroquery.utils.tap.core import TapPlus

SOHO = TapPlus(url="https://ssa.esac.esa.int/ssa-sl-tap/tap")
tables = SOHO.load_tables()#only_names=True)

for table in tables:
    print(table.get_qualified_name())

for table in tables:
    if table.get_qualified_name().startswith("soho.v_"):
        print(table.get_qualified_name())

INFO: Retrieving tables... [astroquery.utils.tap.core]
INFO: Parsing tables... [astroquery.utils.tap.core]
INFO: Done. [astroquery.utils.tap.core]
public.dual
soho.campaign
soho.cds_observation
soho.celias_observation
soho.coronal_image
soho.costep_observation
soho.eit_observation
soho.erne_observation
soho.full_disk_solar_image
soho.golf_observation
soho.instrument
soho.lasco_observation
soho.lyra_observation
soho.mdi_observation
soho.mv_observation
soho.observation
soho.observatory
soho.observing_mode
soho.science_object
soho.science_slit
soho.study
soho.sumer_observation
soho.swan_observation
soho.swap_observation
soho.uvcs_observation
soho.virgo_observation
tap_config.coord_sys
tap_config.properties
tap_schema.columns
tap_schema.key_columns
tap_schema.keys
tap_schema.schemas
tap_schema.tables


From the above list of tables, the helpful ones are those beginning `soho.` and the ones meant for users begin with `soho.v_`, where the 'v' means 'view'. The view tables contain the same information as the other `soho.` tables but the columns in those tables might be grouped slightly differently, and the columns containing information meant for internal database use have been taken out since they are of no use to the end-user.

Also from [Astroquery TAP page](https://astroquery.readthedocs.io/en/latest/utils/tap.html), load a specific table and print out the column names. 

In [7]:
table = SOHO.load_table('soho.observation')

print('\n Table: \n', table, '\n')

print('Column names: ')
for column in (table.columns):
    print(column.name)


 Table: 
 TAP Table name: soho.observation
Description: None
Size (bytes): 5867741184
Num. columns: 17 

Column names: 
begin_date
calibrated
campaign_oid
end_date
file_format
file_name
file_path
file_size
observation_oid
observation_type
observing_mode_oid
processing_level
science_objective
science_object_oid
science_slit_oid
study_oid
wavelength_range


## Querying Tables - accessing metadata

So far we have only been loading metadata of the metadata, so we need to make an actual query to get the values:

From [AppDynamics](https://docs.appdynamics.com/display/PRO45/ADQL+Queries) rather than [Astronomical Data Query Language](https://www.ivoa.net/documents/ADQL/2.0), but still usefully put:

`SELECT * | {[DISTINCT] expression [AS alias] [, expression [AS alias]]...}
FROM event_type
[WHERE condition_expression]
[SINCE timevalue [UNTIL timevalue]]
[HAVING condition_expression]
[ORDER BY {field_name | alias} [ASC | DESC] [, {field_name | alias} [ASC | DESC]]...]
[LIMIT integer [, integer]...]`

While the SELECT and FROM clauses are required, all other clauses are optional. Type spaces around each keyword. 

### Whole table

To see all the columns and first 2000 rows (TAP+ has a synchronous request limit of 2000) of a table:

In [8]:
results = SOHO.launch_job("SELECT * FROM soho.dataset")
r = results.get_results()
r

HTTPError: Error 400: 
Cannot parse query 'SELECT  TOP 2000 * FROM soho.dataset' for job '1793266896060OPE': 1 unresolved identifiers: dataset [l.1 c.25 - l.1 c.37] !
 - Unknown table "soho.dataset" !


### Using WHERE to restrict results

To get a list of all datasets that include FGM, we need to query the v_dataset table for the dataset_id and add the WHERE statement to restrict the results to those that contain 'FGM'. Remember that the percentage sign (%) is the wildcard, but it doesn't need to be URL encoded here.

In [ ]:
results = CSA.launch_job("SELECT dataset_id FROM csa.v_dataset WHERE dataset_id like '%FGM%'")
r = results.get_results()
r

dataset_id
object
CM_CD_FGM_CR_ICD_UG
C4_PP_FGM
CL_SP_FGM
C4_CQ_FGM_CAVF
C4_CQ_FGM_GAPF
C4_CQ_FGM_VALF
C4_CP_FGM_5VPS_ISR2
C4_CP_FGM_5VPS
C4_CP_FGM_FULL_ISR2


### An inventory file request:

Request (SELECT) 5 columns FROM csa.v_dataset_inventory table WHERE the dataset_id is 'C1_CP_FGM_SPIN' AND start_time is *before* 1 May 2002 AND end_time is *after* 1 April 2002 and ORDER BY the start time column, and print the results, as text or as an [astropy.table](https://docs.astropy.org/en/stable/table/index.html?highlight=astropy.table):

In [ ]:
results = SOHO.launch_job("SELECT dataset_id,start_time,end_time,num_instances,inventory_version "
                         "FROM soho.v_dataset_inventory "
                         "WHERE dataset_id='C1_CP_FGM_SPIN' "
                         "AND start_time<='2002-05-01T00:00:00' " 
                         "AND end_time>='2002-04-01T00:00:00' "
                         "ORDER BY start_time")

# Either print the results as text, or as an astropy.table (which can be easier to read)
#print(results.get_results())

r = results.get_results()
r

dataset_id,start_time,end_time,num_instances,inventory_version
object,object,object,int32,int32
C1_CP_FGM_SPIN,2002-03-30T18:17:57.879Z,2002-04-01T00:00:00.000Z,0,10
C1_CP_FGM_SPIN,2002-04-01T00:00:00.000Z,2002-04-01T00:21:04.972Z,0,10
C1_CP_FGM_SPIN,2002-04-01T00:21:04.972Z,2002-04-01T00:28:51.002Z,117,10
C1_CP_FGM_SPIN,2002-04-01T00:28:51.002Z,2002-04-01T00:35:16.683Z,0,10
C1_CP_FGM_SPIN,2002-04-01T00:35:16.683Z,2002-04-01T03:36:48.128Z,2704,10
C1_CP_FGM_SPIN,2002-04-01T03:36:48.128Z,2002-04-01T18:41:24.702Z,0,10
C1_CP_FGM_SPIN,2002-04-01T18:41:24.702Z,2002-04-01T18:48:50.646Z,112,10
C1_CP_FGM_SPIN,2002-04-01T18:48:50.646Z,2002-04-01T18:55:20.346Z,0,10
C1_CP_FGM_SPIN,2002-04-01T18:55:20.346Z,2002-04-02T06:44:49.979Z,10534,10
